# Mastering Machine Learning Workflows
## Pipelines, Hyperparameter Tuning, and MLflow

Welcome! This notebook is designed to take you from writing 'loose' machine learning code to building professional, robust, and reproducible ML pipelines.

### Prerequisites
* Basic Python (Lists, Dictionaries, Functions)
* Familiarity with `pandas` and `numpy`
* Basic understanding of Machine Learning (Features vs. Targets)

---

## Part 1: Why Machine Learning Workflows Need Structure

In a typical ML project, you don't just 'fit' a model. You have to:
1. Handle missing values.
2. Scale numbers (like Age or Fare).
3. Encode text (like 'Male'/'Female' to 0/1).
4. Split data into Training and Testing sets.

### The Danger: Data Leakage
**Data Leakage** happens when information from the test set 'leaks' into the training process. For example, if you calculate the average age of the *entire* dataset to fill missing values before splitting, your training model now 'knows' something about the test set. This leads to overly optimistic results that fail in the real world.

### The Solution: Structure
We need a way to ensure that every transformation we apply to the training data is applied *exactly the same way* to the test data without looking at the test data's statistics.

In [4]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load the Titanic dataset
df = sns.load_dataset('titanic')

# Quick look at the data
display(df.head())
print(f"\nDataset Shape: {df.shape}")

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True



Dataset Shape: (891, 15)


## Part 2: Scikit-Learn Pipelines

### What is a Pipeline?
Think of a Pipeline as an **assembly line** in a factory. Raw materials go in one end, and a finished product comes out the other.

**ASCII Intuition:**
```text
[ Raw Data ] -> [ Imputer ] -> [ Scaler ] -> [ Classifier ] -> [ Prediction ]
```

### Why use them?
1. **Clean Code:** You only call `.fit()` and `.predict()` once on the pipeline.
2. **Consistency:** It prevents data leakage by ensuring transformers only see the training data statistics.
3. **Deployment:** You can save the entire pipeline (processing + model) as a single file.

### Manual Workflow (The 'Difficult' Way)
Before we use Pipelines, let's see how messy it is to do things manually.

In [6]:
# Simple manual example with one numeric column
X = df[['age']].fillna(df['age'].median())
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 1: Scale training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Step 2: Scale test data (Using parameters FROM training set!)
X_test_scaled = scaler.transform(X_test)

# Step 3: Train model
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

print(f"Accuracy: {model.score(X_test_scaled, y_test):.2f}")

Accuracy: 0.59


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

### Creating a Basic Pipeline

Now, let's do the same thing using the `Pipeline` class. The Pipeline takes a list of tuples: `('name', object)`.

* **'scaler'**: The step name.
* **StandardScaler()**: The transformer.
* **'model'**: The step name.
* **LogisticRegression()**: The estimator.

In [6]:
basic_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

# We pass RAW training data (pre-filled for this simple example)
basic_pipe.fit(X_train, y_train)

# Predictions
preds = basic_pipe.predict(X_test)
print(f"Pipeline Accuracy: {accuracy_score(y_test, preds):.2f}")

Pipeline Accuracy: 0.59


## Part 3: ColumnTransformer

In the Titanic dataset, we have:
* **Numerical columns** (Age, Fare): Need scaling.
* **Categorical columns** (Sex, Embarked): Need One-Hot Encoding.

We can't apply the same rule to every column. This is where `ColumnTransformer` comes in.

**ASCII Diagram:**
```text
Column A (Num)  -----> [ Scaler ] ----\
                                       |---> [ Combined Vector ] ---> [ Model ]
Column B (Cat)  -----> [ Encoder ] ----/
```

In [7]:
# Select features
features = ['age', 'fare', 'sex', 'embarked']
X = df[features]
y = df['survived']

# Define preprocessing for numerical data
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Define preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, ['age', 'fare']),
        ('cat', categorical_transformer, ['sex', 'embarked'])
    ])

# Create a full production-style pipeline
clf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Split and train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf_pipeline.fit(X_train, y_train)

print(f"Full Pipeline Test Score: {clf_pipeline.score(X_test, y_test):.4f}")

Full Pipeline Test Score: 0.7542


### 💡 Interview Question
**Q: Why do we use `fit_transform` on training data but only `transform` on test data?**
**A:** We want to learn the parameters (mean, variance, etc.) only from the training data. Applying `transform` on the test data using those training parameters ensures that our evaluation reflects how the model will perform on truly unseen data.

### 🚩 Common Mistake
Encoding labels *before* splitting the data. This allows the encoder to 'see' categories in the test set that might not be in the training set, leading to leakage or errors during prediction.

## Part 4: Cross Validation

### Why Train/Test Split is Not Enough
A single split might be 'lucky' or 'unlucky'. If your test set happens to have all the easy-to-predict cases, your accuracy will look great but won't be realistic.

### The Solution: K-Fold Cross Validation
We split the data into **K** groups (folds). We train on K-1 folds and test on the remaining fold. We repeat this until every fold has been used as a test set.

**Visualizing 5-Fold CV:**
```text
Iteration 1: [ TEST ] [ Train ] [ Train ] [ Train ] [ Train ]
Iteration 2: [ Train ] [ TEST ] [ Train ] [ Train ] [ Train ]
Iteration 3: [ Train ] [ Train ] [ TEST ] [ Train ] [ Train ]
Iteration 4: [ Train ] [ Train ] [ Train ] [ TEST ] [ Train ]
Iteration 5: [ Train ] [ Train ] [ Train ] [ Train ] [ TEST ]
```

### Discussion Question
*If we have 1,000 rows and use 10-fold CV, how many rows are in each test fold?*

In [ ]:
from sklearn.model_selection import cross_val_score

# We use the full pipeline with cross_val_score
# This automatically handles the internal 'fit' and 'transform' for each fold!
scores = cross_val_score(clf_pipeline, X, y, cv=5)

print(f"Scores per fold: {scores}")
print(f"Mean Accuracy: {scores.mean():.4f}")
print(f"Standard Deviation: {scores.std():.4f}")

## Part 5: Hyperparameter Tuning

### Parameters vs. Hyperparameters
*   **Parameters:** Learned by the model during training (e.g., weights in regression).
*   **Hyperparameters:** Settings you choose *before* training to control how the model learns.

#### RandomForest Hyperparameters:
1.  `n_estimators`: Number of trees. More is usually better but slower.
2.  `max_depth`: How deep trees grow. Too deep = Overfitting. Too shallow = Underfitting.
3.  `min_samples_split`: Minimum samples required to split a node.

### GridSearchCV (The Exhaustive Search)
Grid Search tries **every possible combination** of parameters you provide.

**Grid Table Example:**

| n_estimators | max_depth | Combination |
| :--- | :--- | :--- |
| 100 | 5 | (100, 5) |
| 100 | 10 | (100, 10) |
| 200 | 5 | (200, 5) |
| 200 | 10 | (200, 10) |

In [9]:
from sklearn.model_selection import GridSearchCV

# In a Pipeline, we access parameters using 'stepname__parametername'
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 5, 10],
    'classifier__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(clf_pipeline, param_grid, cv=3, verbose=1, n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

Fitting 3 folds for each of 18 candidates, totalling 54 fits
Best Parameters: {'classifier__max_depth': 5, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 50}
Best CV Score: 0.7964


### RandomizedSearchCV (The Efficient Search)
Grid Search becomes very slow if you have many parameters. **Randomized Search** picks a fixed number of random combinations. Usually, it finds a 'good enough' result much faster.

In [10]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    'classifier__n_estimators': randint(50, 500),
    'classifier__max_depth': [None, 5, 10, 20],
    'classifier__min_samples_leaf': randint(1, 5)
}

random_search = RandomizedSearchCV(clf_pipeline, param_dist, n_iter=10, cv=3, random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

# Convert results to DataFrame to visualize
results_df = pd.DataFrame(random_search.cv_results_)
display(results_df[['param_classifier__n_estimators', 'param_classifier__max_depth', 'mean_test_score']].sort_values('mean_test_score', ascending=False).head())

,param_classifier__n_estimators,param_classifier__max_depth,mean_test_score
9,493,5,0.786542
3,264,5,0.785141
7,393,None,0.782358
8,343,20,0.780963
5,409,None,0.780928


## Part 6: MLflow - Tracking Your Experiments

### Why MLflow?
Imagine you run 50 experiments. You change the `max_depth` of your forest, you try different scalers, and you remove different columns.

**How do you remember which settings gave you 87% accuracy vs 82%?**

MLflow acts as a **Laboratory Notebook**. It automatically records:
* **Parameters:** Your settings (e.g., n_estimators=100).
* **Metrics:** Your results (e.g., Accuracy, F1 Score).
* **Artifacts:** Your files (e.g., Confusion Matrix plots).
* **Models:** The actual saved model file.

### Installing MLflow in Colab

In [ ]:
!pip install mlflow --quiet

### Logging a Basic Run
We use `mlflow.start_run()` as a context manager (using `with`). Everything inside that block is recorded to that specific 'experiment'.

In [11]:

import mlflow
import mlflow.sklearn

# Set the experiment name
mlflow.set_experiment("Titanic_Survival_Analysis")

with mlflow.start_run(run_name="Baseline_RandomForest"):
    # 1. Log Parameters
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)

    # 2. Train and Predict
    clf_pipeline.fit(X_train, y_train)
    acc = clf_pipeline.score(X_test, y_test)

    # 3. Log Metrics
    mlflow.log_metric("accuracy", acc)

    # 4. Log the Model
    mlflow.sklearn.log_model(
        sk_model=clf_pipeline,
        name="model",
        skops_trusted_types=[
            "numpy.dtype",
            "sklearn.tree._tree.Tree"
        ]
    )

    print(f"Run logged successfully with accuracy: {acc}")


Run logged successfully with accuracy: 0.7541899441340782


## Part 7: End-to-End Project (The Professional Way)

Now, let's combine **Pipelines + Hyperparameter Tuning + MLflow Tracking** into one final production-style workflow.

In [12]:
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

def run_titanic_experiment(n_estimators, max_depth, run_name):
    with mlflow.start_run(run_name=run_name):

        # 1. Define the pipeline
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42
        )

        full_pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", model)
        ])

        # 2. Train
        full_pipe.fit(X_train, y_train)

        # 3. Evaluate
        y_pred = full_pipe.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        # 4. Log parameters and metrics
        mlflow.log_params({
            "n_estimators": n_estimators,
            "max_depth": max_depth
        })
        mlflow.log_metric("accuracy", acc)

        # 5. Create and log the confusion matrix
        fig, ax = plt.subplots(figsize=(6, 5))

        ConfusionMatrixDisplay.from_estimator(
            full_pipe,
            X_test,
            y_test,
            ax=ax
        )

        ax.set_title(f"Confusion Matrix: {run_name}")
        fig.tight_layout()

        # Use a unique filename for each run
        artifact_path = f"confusion_matrix_{run_name}.png"
        fig.savefig(artifact_path, bbox_inches="tight")
        mlflow.log_artifact(artifact_path)

        plt.close(fig)

        # 6. Log the model with explicitly trusted types
        mlflow.sklearn.log_model(
            sk_model=full_pipe,
            name="titanic_model",
            skops_trusted_types=[
                "numpy.dtype",
                "sklearn.tree._tree.Tree"
            ]
        )

        print(f"{run_name}: accuracy = {acc:.4f}")
        return acc


# Run experiments
acc_small = run_titanic_experiment(50, 5, "Small_Forest")
acc_large = run_titanic_experiment(200, 10, "Large_Forest")
acc_deep = run_titanic_experiment(500, None, "Deep_Forest")


Small_Forest: accuracy = 0.7821
Large_Forest: accuracy = 0.7542
Deep_Forest: accuracy = 0.7374


## Part 8: Summary Table

| Concept | Purpose | Key Tool |
| :--- | :--- | :--- |
| **Data Leakage** | Prevent test info from leaking into training | `train_test_split` (correctly) |
| **Pipeline** | Automate assembly line of ML steps | `sklearn.pipeline.Pipeline` |
| **ColumnTransformer** | Handle different data types (Num vs Cat) | `sklearn.compose.ColumnTransformer` |
| **Cross-Validation** | Get a stable, reliable accuracy score | `cross_val_score` |
| **GridSearch** | Exhaustively find best hyperparameters | `GridSearchCV` |
| **MLflow** | Track history of all experiments | `mlflow.log_metric/param` |

## Part 9: Practice Exercises

### Easy
1. **Pipeline Basics**: Create a pipeline with a `SimpleImputer` and a `LogisticRegression` model.
2. **Scaling**: Use a pipeline to scale the 'fare' column and predict 'survived'.
3. **MLflow logging**: Start an MLflow run and log a single parameter called 'learning_rate' with a value of 0.01.

### Medium
4. **Column Selection**: Use `ColumnTransformer` to apply `StandardScaler` to 'age' and 'pclass', while ignoring other columns.
5. **Cross-Validation**: Calculate the 10-fold cross-validation score for a RandomForest model on the Titanic dataset.
6. **GridSearch**: Run a `GridSearchCV` on a Logistic Regression model to find the best `C` value (try [0.1, 1, 10]).
7. **MLflow Artifacts**: Modify the training code to save and log a feature importance plot as an artifact.

### Hard
8. **Complex Pipeline**: Build a pipeline that includes an imputer, a custom feature engineer (e.g., combining SibSp and Parch), and a GradientBoostingClassifier.
9. **RandomSearch vs GridSearch**: Run both on the same pipeline and compare the time taken vs. the final accuracy score.
10. **Full Cycle**: Write a function that takes a list of features, performs a RandomizedSearch, and logs the results to a new MLflow experiment named 'Final_Challenge'.

## Part 10: Interview Preparation

1. **What is Data Leakage?**
   *Answer:* When information from outside the training dataset is used to create the model. Common in preprocessing if statistics are calculated on the whole set before splitting.

2. **Why use a Pipeline instead of just calling functions?**
   *Answer:* Pipelines ensure that the same transformations are applied to train and test sets, encapsulate the workflow, and prevent leakage.

3. **Difference between GridSearchCV and RandomizedSearchCV?**
   *Answer:* Grid search tries every combination (exhaustive). Random search samples a fixed number of combinations (efficient).

4. **What does the `cv` parameter in GridSearchCV do?**
   *Answer:* It determines the number of folds for cross-validation.

5. **What is an MLflow Artifact?**
   *Answer:* Any output file (images, models, CSVs) generated by a run that you want to save.

6. **How do you access the best model after a GridSearch?**
   *Answer:* Use the `.best_estimator_` attribute.

7. **Why scale data before using Logistic Regression?**
   *Answer:* It helps the optimization algorithm (Gradient Descent) converge faster.

8. **What is K-Fold Cross Validation?**
   *Answer:* Splitting data into K parts, using each part once as a test set and others as training.

9. **Explain underfitting vs overfitting in the context of `max_depth`.**
   *Answer:* High depth can lead to overfitting (learning noise). Low depth can lead to underfitting (too simple).

10. **What is One-Hot Encoding?**
    *Answer:* Converting categorical text into binary columns (0 or 1).

*(Note: 10 more questions included in the teacher's guide version of this notebook)*